# SMOTE for Images - Custom Dataset Integration

This notebook demonstrates how to integrate custom datasets with the SMOTE Image Synthesis pipeline.

In [ ]:
import torch
import numpy as np
import sys
import os
from pathlib import Path
import tempfile
from PIL import Image

# Add the project root to Python path
sys.path.insert(0, os.path.dirname(os.getcwd()))

# Import pipeline components
from smote_image_synthesis.data.models import PipelineConfig
from smote_image_synthesis.encoders.resnet_encoder import ResNetEncoder
from smote_image_synthesis.decoders.autoencoder_decoder import AutoencoderDecoder
from smote_image_synthesis.smote.constrained_smote import ConstrainedSMOTE
from smote_image_synthesis.quality.assessor import QualityAssessor
from smote_image_synthesis.pipeline import SynthesisPipeline

## Create a Mock Custom Dataset

For demonstration purposes, we'll create a mock custom dataset with different classes.

In [ ]:
def create_mock_dataset(dataset_dir):
    """Create a mock dataset with different classes for demonstration."""
    dataset_path = Path(dataset_dir)
    dataset_path.mkdir(exist_ok=True)
    
    # Create class directories
    classes = ['cats', 'dogs', 'birds']
    class_paths = {}
    
    for class_name in classes:
        class_path = dataset_path / class_name
        class_path.mkdir(exist_ok=True)
        class_paths[class_name] = class_path
    
    # Generate mock images for each class
    image_size = (64, 64)
    
    # Cats: Circular patterns
    for i in range(15):  # 15 cat images (minority class)
        img = Image.new('RGB', image_size, color='white')
        pixels = img.load()
        center_x, center_y = image_size[0] // 2, image_size[1] // 2
        radius = min(image_size) // 4
        
        for x in range(image_size[0]):
            for y in range(image_size[1]):
                if (x - center_x)**2 + (y - center_y)**2 <= radius**2:
                    pixels[x, y] = (255, 100, 100)  # Light red circle
        
        img.save(class_paths['cats'] / f'cat_{i:03d}.png')
    
    # Dogs: Square patterns
    for i in range(30):  # 30 dog images (majority class)
        img = Image.new('RGB', image_size, color='white')
        pixels = img.load()
        start_x = image_size[0] // 4
        start_y = image_size[1] // 4
        size = min(image_size) // 2
        
        for x in range(start_x, start_x + size):
            for y in range(start_y, start_y + size):
                pixels[x, y] = (100, 255, 100)  # Light green square
        
        img.save(class_paths['dogs'] / f'dog_{i:03d}.png')
    
    # Birds: Diagonal patterns
    for i in range(8):  # 8 bird images (minority class)
        img = Image.new('RGB', image_size, color='white')
        pixels = img.load()
        
        for offset in range(5):
            for x in range(image_size[0]):
                y = x + offset
                if 0 <= y < image_size[1]:
                    pixels[x, y] = (100, 100, 255)  # Light blue diagonal
                y = x - offset
                if 0 <= y < image_size[1]:
                    pixels[x, y] = (100, 100, 255)  # Light blue diagonal
        
        img.save(class_paths['birds'] / f'bird_{i:03d}.png')
    
    print(f"Mock dataset created at: {dataset_path}")
    print(f"Classes and counts:")
    for class_name in classes:
        count = len(list(class_paths[class_name].glob('*.png')))
        print(f"  {class_name}: {count} images")
    
    return dataset_path, classes

# Create mock dataset
mock_dataset_dir = "./mock_dataset"
dataset_path, class_names = create_mock_dataset(mock_dataset_dir)

## Load and Preprocess Custom Dataset

Implement functions to load and preprocess the custom dataset.

In [ ]:
def load_custom_dataset(dataset_path, class_names):
    """Load custom dataset from directory structure."""
    images = []
    labels = []
    
    # Map class names to numeric labels
    class_to_label = {name: idx for idx, name in enumerate(class_names)}
    
    for class_name in class_names:
        class_path = Path(dataset_path) / class_name
        image_files = list(class_path.glob('*.png'))
        
        print(f"Loading {len(image_files)} images from {class_name} class")
        
        for img_file in image_files:
            # Load and preprocess image
            img = Image.open(img_file).convert('RGB')
            
            # Resize to standard size
            img = img.resize((64, 64))
            
            # Convert to tensor
            img_array = np.array(img).astype(np.float32)  # Shape: (H, W, C)
            img_tensor = torch.from_numpy(img_array).permute(2, 0, 1)  # Shape: (C, H, W)
            
            # Normalize manually (using ImageNet stats)
            normalize_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            normalize_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            img_tensor = (img_tensor / 255.0 - normalize_mean) / normalize_std
            
            images.append(img_tensor)
            labels.append(class_to_label[class_name])
    
    images_tensor = torch.stack(images)
    labels_array = np.array(labels)
    
    print(f"Loaded dataset: {images_tensor.shape}")
    print(f"Class distribution: {np.bincount(labels_array)}")
    
    return images_tensor, labels_array

# Load the custom dataset
images, labels = load_custom_dataset(dataset_path, class_names)

## Set Up Pipeline Configuration

Configure the pipeline for the custom dataset.

In [ ]:
# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create pipeline configuration
config = PipelineConfig(
    config_name="custom_dataset_example",
    output_dir="./custom_dataset_output"
)

# Adjust configuration for custom dataset
config.encoder_config.architecture = 'resnet18'
config.encoder_config.embedding_dim = 128
config.encoder_config.pretrained = False  # Use non-pretrained for faster execution
config.decoder_config.image_shape = (3, 64, 64)  # Match our dataset
config.decoder_config.decoder_type = 'autoencoder'
config.smote_config.k_neighbors = 3
config.quality_config.metrics = ['mse', 'mae']  # Use simple metrics

print("Pipeline configuration:")
print(f"- Encoder: {config.encoder_config.architecture}")
print(f"- Embedding dimension: {config.encoder_config.embedding_dim}")
print(f"- Decoder type: {config.decoder_config.decoder_type}")
print(f"- SMOTE k_neighbors: {config.smote_config.k_neighbors}")

## Create and Configure Pipeline Components

Set up the encoder, decoder, SMOTE, and quality assessor.

In [ ]:
# Create encoder
encoder = ResNetEncoder(
    architecture=config.encoder_config.architecture,
    embedding_dim=config.encoder_config.embedding_dim,
    pretrained=config.encoder_config.pretrained,
    device=device
)

# Create decoder
decoder = AutoencoderDecoder(
    embedding_dim=config.encoder_config.embedding_dim,
    image_shape=config.decoder_config.image_shape,
    device=device
)

# Create SMOTE
smote = ConstrainedSMOTE(
    k_neighbors=config.smote_config.k_neighbors,
    use_clustering=config.smote_config.use_clustering,
    clustering_method=config.smote_config.cluster_method,
    random_state=config.seed
)

# Create quality assessor
quality_assessor = QualityAssessor(
    metrics=config.quality_config.metrics,
    device=device
)

print("Pipeline components created successfully!")

## Create and Fit Pipeline

Create the synthesis pipeline and fit it on the custom dataset.

In [ ]:
# Create pipeline
pipeline = SynthesisPipeline(
    encoder=encoder,
    decoder=decoder,
    smote=smote,
    quality_assessor=quality_assessor
)

print("Fitting pipeline on custom dataset...")
# Fit pipeline (this encodes images and fits SMOTE)
pipeline.fit(images, labels, train_decoder=False)

print("Pipeline fitted successfully!")

## Analyze Class Imbalance

Check the class distribution before and after synthetic generation.

In [ ]:
# Analyze original class distribution
unique, counts = np.unique(labels, return_counts=True)
original_distribution = dict(zip(unique, counts))

print("Original class distribution:")
for class_idx, count in original_distribution.items():
    class_name = class_names[class_idx]
    print(f"  {class_name} (class {class_idx}): {count} samples")

# Calculate how many samples to generate for balance
max_class_count = max(counts)
target_distribution = {class_idx: max_class_count for class_idx in unique}

print(f"\nTarget distribution for balance: {max_class_count} samples per class")
print("Samples to generate per class:")
for class_idx, orig_count in original_distribution.items():
    to_generate = max_class_count - orig_count
    class_name = class_names[class_idx]
    print(f"  {class_name} (class {class_idx}): {to_generate} samples")

## Generate Synthetic Images

Generate synthetic images to balance the dataset.

In [ ]:
# Calculate how many synthetic images to generate
total_to_generate = sum(max_class_count - count for count in counts)
print(f"Generating {total_to_generate} synthetic images to balance the dataset")

# Generate synthetic images
print("Generating synthetic images...")
synthetic_images, synthetic_labels = pipeline.generate_synthetic_images(total_to_generate)

print(f"Generated {len(synthetic_images)} synthetic images")
if len(synthetic_labels) > 0:
    unique_syn, counts_syn = np.unique(synthetic_labels, return_counts=True)
    synthetic_distribution = dict(zip(unique_syn, counts_syn))
    
    print("Synthetic images distribution:")
    for class_idx, count in synthetic_distribution.items():
        class_name = class_names[class_idx]
        print(f"  {class_name} (class {class_idx}): {count} samples")

# Combine original and synthetic data
combined_images = torch.cat([images, synthetic_images], dim=0) if len(synthetic_images) > 0 else images
combined_labels = np.concatenate([labels, synthetic_labels]) if len(synthetic_labels) > 0 else labels

# Analyze combined distribution
unique_combined, counts_combined = np.unique(combined_labels, return_counts=True)
combined_distribution = dict(zip(unique_combined, counts_combined))

print(f"\nCombined dataset distribution:")
for class_idx, count in combined_distribution.items():
    class_name = class_names[class_idx]
    print(f"  {class_name} (class {class_idx}): {count} samples")

## Evaluate Quality of Synthetic Images

Assess the quality of the generated synthetic images.

In [ ]:
# Evaluate quality if we have synthetic images
if len(synthetic_images) > 0:
    print("\nEvaluating quality of synthetic images...")
    
    # Use a subset for evaluation to avoid memory issues
    n_eval = min(10, len(images), len(synthetic_images))
    eval_real = images[:n_eval]
    eval_synthetic = synthetic_images[:n_eval]
    
    quality_results = pipeline.evaluate_quality(eval_synthetic, eval_real)
    
    print("Quality Results:")
    for metric, value in quality_results.items():
        if isinstance(value, dict):
            for sub_metric, sub_value in value.items():
                print(f"  {metric}.{sub_metric}: {sub_value:.6f}")
        else:
            print(f"  {metric}: {value:.6f}")
else:
    print("\nNo synthetic images generated to evaluate.")

## Working with Real Datasets

Here's a template for integrating real datasets with the pipeline.

In [ ]:
def integrate_real_dataset(dataset_path, class_names, target_size=(64, 64)):
    """
    Template function for integrating real datasets with the SMOTE pipeline.
    
    Args:
        dataset_path: Path to the dataset directory
        class_names: List of class names
        target_size: Target size for images
    
    Returns:
        images_tensor: Tensor of images
        labels_array: Array of labels
    """
    images = []
    labels = []
    
    # Map class names to numeric labels
    class_to_label = {name: idx for idx, name in enumerate(class_names)}
    
    for class_name in class_names:
        class_path = Path(dataset_path) / class_name
        if not class_path.exists():
            print(f"Warning: Class directory {class_path} does not exist")
            continue
            
        # Supported image extensions
        extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
        image_files = []
        for ext in extensions:
            image_files.extend(class_path.glob(f'*{ext}'))
            image_files.extend(class_path.glob(f'*{ext.upper()}'))
        
        print(f"Loading {len(image_files)} images from {class_name} class")
        
        for img_file in image_files:
            try:
                # Load and preprocess image
                img = Image.open(img_file).convert('RGB')
                
                # Resize to target size
                img = img.resize(target_size)
                
                # Convert to tensor
                img_array = np.array(img).astype(np.float32)  # Shape: (H, W, C)
                img_tensor = torch.from_numpy(img_array).permute(2, 0, 1)  # Shape: (C, H, W)
                
                # Normalize using ImageNet stats
                normalize_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
                normalize_std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
                img_tensor = (img_tensor / 255.0 - normalize_mean) / normalize_std
                
                images.append(img_tensor)
                labels.append(class_to_label[class_name])
            
            except Exception as e:
                print(f"Error loading {img_file}: {e}")
    
    if not images:
        raise ValueError("No images could be loaded from the dataset")
    
    images_tensor = torch.stack(images)
    labels_array = np.array(labels)
    
    print(f"Loaded dataset: {images_tensor.shape}")
    print(f"Class distribution: {np.bincount(labels_array)}")
    
    return images_tensor, labels_array

print("Template function 'integrate_real_dataset' defined.")
print("This function can be used to load real datasets with the expected directory structure:")
print("dataset_path/")
print("├── class1/")
print("│   ├── image1.jpg")
print("│   ├── image2.png")
print("│   └── ...")
print("├── class2/")
print("│   ├── image1.jpg")
print("│   ├── image2.png")
print("│   └── ...")
print("└── ...")

## Summary

This notebook demonstrated how to integrate custom datasets with the SMOTE Image Synthesis pipeline:

1. Created a mock dataset with class imbalance
2. Implemented functions to load and preprocess custom datasets
3. Configured the pipeline for the custom dataset
4. Analyzed the class imbalance in the dataset
5. Generated synthetic images to balance the dataset
6. Evaluated the quality of synthetic images
7. Provided a template for integrating real datasets

The SMOTE Image Synthesis pipeline can effectively handle custom datasets with class imbalance by generating synthetic images that help balance the class distribution while maintaining quality.